# Notebook 04 — Normalización del experimento de referencia

Implementación pública alineada con la normalización usada por los modelos M1–M5 reportados en el manuscrito.

- Periodo operativo: **2013–2025**.
- Periodo para estimar parámetros Landsat: **2013–2019**.
- Índices espectrales: robust min–max **P2–P98**, recortado a [0,1].
- LST: z-score global con media y desviación estándar de 2013–2019.
- Los parámetros se congelan para años posteriores; VALIDATION y TEST no intervienen en su estimación.
- La carpeta histórica conserva el nombre `Barranquilla_LST_2013_2026_Modeling`; 2026 está fuera del alcance público actual.


In [ ]:
from pathlib import Path
import json, shutil
import numpy as np
import pandas as pd
import rasterio
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling")
V6_DIR = PROJECT_DIR / "03_annual_raw_composites" / "v6_common_valid_mask"
OUT_DIR = PROJECT_DIR / "05_normalized_products" / "v6_train_only_mixed_strategy"
PARAMS_DIR = OUT_DIR / "_normalization_params"
REPORT_DIR = PROJECT_DIR / "00_documentation" / "quality_control" / "04_train_only_normalization_2013_2025"
TRAIN_SPATIAL_MASK = PROJECT_DIR / "01_aoi_reference" / "train_spatial_mask" / "train_spatial_mask.tif"
MODEL_DOMAIN_REFERENCE = PROJECT_DIR / "01_aoi_reference" / "reference_grid" / "model_domain_reference.tif"
for d in (OUT_DIR, PARAMS_DIR, REPORT_DIR): d.mkdir(parents=True, exist_ok=True)

YEARS_ALL = list(range(2013, 2026))
YEARS_TRAIN = list(range(2013, 2020))
SPECTRAL_VARS = ["NDVI", "NDMI", "NDBI", "UI", "SAVI", "BSI"]
NOBS_VARS = ["n_obs_indices", "n_obs_LST"]
ROBUST_P_LOW, ROBUST_P_HIGH = 2, 98
NODATA_FLOAT = -9999.0
EXPECTED_LST_MEAN = 38.48322677612305
EXPECTED_LST_STD = 4.032179355621338
EXPECTED_N_TRAIN = 4301427

def v6_path(var, year):
    if var == "LST": return V6_DIR / f"EXP10Bv6_LST_median_{year}.tif"
    if var in SPECTRAL_VARS: return V6_DIR / f"EXP10Bv6_{var}_median_{year}.tif"
    if var == "n_obs_indices": return V6_DIR / f"EXP10Bv6_n_obs_indices_{year}.tif"
    if var == "n_obs_LST": return V6_DIR / f"EXP10Bv6_n_obs_LST_{year}.tif"
    raise ValueError(var)

def norm_path(var, year):
    if var == "LST": return OUT_DIR / f"EXP10C_LST_zscore_train_{year}.tif"
    return OUT_DIR / f"EXP10C_{var}_robust01_train_{year}.tif"

def read_float(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        nodata, profile = src.nodata, src.profile.copy()
        geom = ((src.height, src.width), src.crs, src.transform)
    if nodata is not None: arr[arr == nodata] = np.nan
    arr[arr == NODATA_FLOAT] = np.nan
    arr[~np.isfinite(arr)] = np.nan
    return arr, profile, geom

with rasterio.open(MODEL_DOMAIN_REFERENCE) as ref:
    ref_geom = ((ref.height, ref.width), ref.crs, ref.transform)
with rasterio.open(TRAIN_SPATIAL_MASK) as src:
    mask = src.read(1); mask_nodata = src.nodata
    mask_geom = ((src.height, src.width), src.crs, src.transform)
if mask_geom != ref_geom: raise ValueError("train_spatial_mask no coincide con la referencia geométrica")
train_spatial = (mask > 0) if mask_nodata is None else ((mask != mask_nodata) & (mask > 0))

for year in YEARS_ALL:
    for var in SPECTRAL_VARS + ["LST"] + NOBS_VARS:
        p = v6_path(var, year)
        if not p.exists(): raise FileNotFoundError(p)
        with rasterio.open(p) as src:
            geom = ((src.height, src.width), src.crs, src.transform)
        if geom != ref_geom: raise ValueError(f"Geometría inconsistente: {p}")

params = {}
rows = []
for var in SPECTRAL_VARS:
    blocks = []
    for year in YEARS_TRAIN:
        arr, _, _ = read_float(v6_path(var, year))
        blocks.append(arr[np.isfinite(arr) & train_spatial].astype("float32"))
    vals = np.concatenate(blocks)
    lo = float(np.nanpercentile(vals, ROBUST_P_LOW)); hi = float(np.nanpercentile(vals, ROBUST_P_HIGH))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo: raise ValueError(f"Rango inválido para {var}")
    params[var] = {"type":"spectral_index","method":"robust_percentile_0_1_train_only","train_years":YEARS_TRAIN,"spatial_mask":str(TRAIN_SPATIAL_MASK),"p_low":ROBUST_P_LOW,"p_high":ROBUST_P_HIGH,"value_low_train":lo,"value_high_train":hi,"range_train":hi-lo,"clip_output_to_0_1":True,"n_train_values":int(vals.size)}
    rows.append({"var":var, **params[var]})

blocks = []
for year in YEARS_TRAIN:
    arr, _, _ = read_float(v6_path("LST", year))
    blocks.append(arr[np.isfinite(arr) & train_spatial].astype("float32"))
vals = np.concatenate(blocks)
mu = float(np.nanmean(vals)); sigma = float(np.nanstd(vals, ddof=0))
params["LST"] = {"type":"target","method":"zscore_train_only","train_years":YEARS_TRAIN,"spatial_mask":str(TRAIN_SPATIAL_MASK),"mean_train":mu,"std_train":sigma,"n_train_values":int(vals.size)}
rows.append({"var":"LST", **params["LST"]})

assert int(vals.size) == EXPECTED_N_TRAIN, (vals.size, EXPECTED_N_TRAIN)
assert np.isclose(mu, EXPECTED_LST_MEAN, atol=1e-7), (mu, EXPECTED_LST_MEAN)
assert np.isclose(sigma, EXPECTED_LST_STD, atol=1e-7), (sigma, EXPECTED_LST_STD)

for year in YEARS_ALL:
    for var in SPECTRAL_VARS + ["LST"]:
        arr, profile, _ = read_float(v6_path(var, year))
        if var == "LST":
            norm = (arr - mu) / sigma
        else:
            lo, hi = params[var]["value_low_train"], params[var]["value_high_train"]
            norm = np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
        out = np.where(np.isfinite(norm), norm, NODATA_FLOAT).astype("float32")
        profile.update(dtype="float32", nodata=NODATA_FLOAT, compress="deflate")
        with rasterio.open(norm_path(var, year), "w", **profile) as dst: dst.write(out, 1)
    for var in NOBS_VARS:
        shutil.copy2(v6_path(var, year), OUT_DIR / f"EXP10C_{var}_{year}.tif")

params_csv = PARAMS_DIR / "normalization_params_train_only_mixed_strategy_2013_2019.csv"
params_json = PARAMS_DIR / "normalization_params_train_only_mixed_strategy_2013_2019.json"
pd.DataFrame(rows).to_csv(params_csv, index=False)
with open(params_json, "w", encoding="utf-8") as f: json.dump(params, f, indent=2, ensure_ascii=False)
print(f"LST mean={mu:.12f} | std={sigma:.12f} | n={vals.size}")
print("Parámetros:", params_json)
